# Vignette 03: Benchmarks

Compare mlsurv's **default hyperparameters** (no tuning) against published results:

1. **JMLR** — Kvamme et al. (2019) / Kvamme & Borgan (2021): SUPPORT, METABRIC, FLCHAIN
2. **scikit-survival docs** — GBSG2, Veterans (mlsurv vs scikit-survival, averaged over 30 seeds)

*Dataset sources*: SUPPORT (Knaus et al. 1995) and METABRIC (Curtis et al. 2012) are used here as the DeepSurv-processed subsets (Katzman et al. 2018); FLCHAIN (Dispenzieri et al. 2012); GBSG2 (Schumacher et al. 1994); Veterans (Kalbfleisch & Prentice 2002). See References.

**Requires the deep extra.** This vignette trains DeepSurv and DeepHit, which need
PyTorch and pycox. These are not part of the base `mlsurv` install — install them with:

```bash
pip install "mlsurv[deep]"
```

*Runtime: ~10–12 minutes on a modern laptop.*

In [ ]:
import contextlib
import io
import os
import time
import warnings
import numpy as np
import pandas as pd
from pycox.datasets import support, metabric, flchain
from sksurv.datasets import load_gbsg2, load_veterans_lung_cancer
from sksurv.preprocessing import encode_categorical

from mlsurv import SurvivalLearner
from mlsurv.exceptions import MethodologicalWarning, SuggestionWarning
from mlsurv.utils import to_structured_array

# These benchmarks rebuild a fresh SurvivalLearner for every dataset/model/seed,
# so mlsurv's per-fit methodological warnings (PH violations, small-sample notes)
# would repeat hundreds of times and bury the tables. Silence them notebook-wide;
# see docs/warnings.md to inspect or re-enable.
warnings.filterwarnings("ignore", category=MethodologicalWarning)
warnings.filterwarnings("ignore", category=SuggestionWarning)

# Results are written to ./output/
os.makedirs("output", exist_ok=True)

## Data loading helpers

In [ ]:
def load_pycox_dataset(loader):
    """Load a pycox dataset into mlsurv format."""
    if loader is flchain:
        df = loader.read_df(processed=False)
        df = (df
              .drop(['chapter', 'rownames'], axis=1, errors='ignore')
              .loc[lambda x: x['creatinine'].notna()]
              .reset_index(drop=True)
              .assign(sex=lambda x: (x['sex'] == 'M').astype(float))
              .rename(columns={'futime': 'duration', 'death': 'event'}))
        for col in df.columns:
            if col not in ['duration', 'event']:
                df[col] = df[col].astype('float32')
    else:
        df = loader.read_df()
    df = df[df['duration'] > 0]
    X = df.drop(columns=['duration', 'event'])
    y = to_structured_array(df['duration'].values, df['event'].values)
    return X, y


def prepare_gbsg2():
    """Preprocess GBSG2 matching sksurv docs: one-hot encode categoricals."""
    X, y_raw = load_gbsg2()
    X = encode_categorical(X)
    return X, to_structured_array(y_raw)


def prepare_veterans():
    """Preprocess Veterans matching sksurv docs: one-hot encode categoricals."""
    X, y_raw = load_veterans_lung_cancer()
    X = encode_categorical(X)
    return X, to_structured_array(y_raw)

## JMLR Benchmarks (C^td, cross-validated)

Kvamme et al. (2019) and Kvamme & Borgan (2021) report 5-fold cross-validated C^td on SUPPORT, METABRIC, and FLCHAIN using 300 random hyperparameter trials per fold. We compare mlsurv's CV C^td using default hyperparameters (no tuning).

| Dataset  | CoxPH | DeepSurv | RSF   | DeepHit |
|----------|-------|----------|-------|---------|
| SUPPORT  | 0.598 | 0.611    | 0.628 | 0.639   |
| METABRIC | 0.628 | 0.636    | 0.649 | 0.675   |
| FLCHAIN  | 0.790 | 0.790    | 0.784 | 0.791   |

*Sources: CoxPH, DeepSurv, RSF from Kvamme et al. (2019, Table 2); DeepHit from Kvamme & Borgan (2021, Table 2).*

In [ ]:
JMLR_TARGETS = {
    'SUPPORT':  {'coxph': 0.598, 'deepsurv': 0.611, 'rsf': 0.628, 'deephit': 0.639},
    'METABRIC': {'coxph': 0.628, 'deepsurv': 0.636, 'rsf': 0.649, 'deephit': 0.675},
    'FLCHAIN':  {'coxph': 0.790, 'deepsurv': 0.790, 'rsf': 0.784, 'deephit': 0.791},
}

# Kvamme one-hot encodes SUPPORT's categoricals (x1-x6) for the classical Cox baseline. 
# The neural nets and RSF keep the raw codes (Kvamme's inputs for those models). 
COX_CATEGORICAL = {'SUPPORT': ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']}

jmlr_models = ['coxph', 'deepsurv', 'rsf', 'deephit']
jmlr_rows = []

for ds_name, loader in [('SUPPORT', support), ('METABRIC', metabric), ('FLCHAIN', flchain)]:
    X, y = load_pycox_dataset(loader)

    # DeepSurv / RSF / DeepHit run on the raw covariates.
    learner = SurvivalLearner(X, y)
    learner.setup(test_fraction=0.2, cv_folds=5, random_state=42)
    results = learner.run(models=['deepsurv', 'rsf', 'deephit'], tune=False,
                          verbose=False, n_importance_repeats=0, bootstrap=False)

    # CoxPH runs on Kvamme's classical-Cox preprocessing: one-hot the nominal
    # categoricals. For METABRIC/FLCHAIN, COX_CATEGORICAL has no entry, so X_cox
    # is X unchanged and those CoxPH rows are identical to a single-pass run.
    cat = COX_CATEGORICAL.get(ds_name)
    X_cox = (pd.get_dummies(X, columns=cat, drop_first=True).astype('float32')
             if cat else X)
    cox_learner = SurvivalLearner(X_cox, y)
    cox_learner.setup(test_fraction=0.2, cv_folds=5, random_state=42)
    results['coxph'] = cox_learner.run(models=['coxph'], tune=False, verbose=False,
                                       n_importance_repeats=0, bootstrap=False)['coxph']

    for model in jmlr_models:
        r = results[model]
        ctd = r.get_metric('ctd', phase='cv', stat='mean')
        target = JMLR_TARGETS[ds_name][model]
        jmlr_rows.append({'dataset': ds_name, 'model': model,
                          'ctd': ctd, 'target': target,
                          'gap': ctd - target if ctd is not None else float('nan')})

### JMLR Summary

In [ ]:
jmlr_df = pd.DataFrame(jmlr_rows)
print("C^td vs JMLR targets (default hyperparameters, no tuning)")
print("-" * 55)
for _, row in jmlr_df.iterrows():
    ctd_str = f"{row['ctd']:.3f}" if pd.notna(row['ctd']) else "N/A"
    print(f"  {row['dataset']:10s} {row['model']:12s}  {ctd_str}  target={row['target']:.3f}  gap={row['gap']:+.3f}")
print(f"\nMean |gap|: {jmlr_df['gap'].abs().mean():.3f}")
jmlr_df.to_csv("output/vig3_jmlr.csv", index=False, float_format="%.6f")

### SUPPORT: DeepSurv and DeepHit need tuning

At default hyperparameters, CoxPH and RSF land on target for SUPPORT, while **DeepSurv (-0.029)** and **DeepHit (-0.039)** still gap. The two neural nets recover their targets after tuning (next section). Kvamme et al. (2019) and Kvamme & Borgan (2021) report performance after 300 trials of random hyperparameter search per model per dataset. Below we provide tuned hyperparameters from 100 Optuna trials each (scoring=ctd, seed=42). To reproduce the tuning, see the cell after the results.

In [ ]:
# Tuned hyperparameters from 100 Optuna trials each (scoring=ctd, seed=42).
# These mirror what `learner.run('deepsurv', n_trials=100)` would find. The
# search space uses n_layers + n_nodes for tuning, but the model constructor
# takes layers=[...] directly, so we translate before passing as model_params.
DEEPSURV_SUPPORT_TUNED = {
    'layers': [48, 48],
    'dropout': 0.32958492226939484,
    'activation': 'relu',
    'weight_decay': 1.660964137194432e-05,
    'batch_size': 256,
    'learning_rate': 0.015152886456951219,
}

DEEPHIT_SUPPORT_TUNED = {
    'layers': [53, 53],
    'dropout': 0.18039799469672485,
    'activation': 'relu',
    'alpha': 0.04197763946319677,
    'sigma': 0.03508106342236002,
    'learning_rate': 0.003089034890190141,
    'weight_decay': 0.008746578044256804,
    'batch_size': 256,
    'num_durations': 258,
}

X_support, y_support = load_pycox_dataset(support)
learner_support = SurvivalLearner(X_support, y_support)
learner_support.setup(test_fraction=0.2, cv_folds=5, random_state=42)

results_tuned = learner_support.run(
    models=['deepsurv', 'deephit'], tune=False, verbose=False,
    model_params={
        'deepsurv': DEEPSURV_SUPPORT_TUNED,
        'deephit': DEEPHIT_SUPPORT_TUNED,
    },
    n_importance_repeats=0, bootstrap=False,
)

# Pull defaults from the earlier benchmark for a side-by-side comparison
defaults_lookup = (jmlr_df[jmlr_df['dataset'] == 'SUPPORT']
                   .set_index('model')['ctd'].to_dict())
target_lookup = JMLR_TARGETS['SUPPORT']

tuned_rows = []
print('SUPPORT: defaults vs tuned (100 Optuna trials, scoring=ctd)')
print('-' * 72)
print(f"  {'Model':10s}  {'Default':>9s}  {'Tuned':>9s}  {'Target':>9s}  "
      f"{'Default gap':>12s}  {'Tuned gap':>10s}")
for model in ['deepsurv', 'deephit']:
    r = results_tuned[model]
    ctd_t = r.get_metric('ctd', phase='cv', stat='mean')
    d, t = defaults_lookup[model], target_lookup[model]
    print(f"  {model:10s}  {d:9.3f}  {ctd_t:9.3f}  {t:9.3f}  "
          f"{d-t:+12.3f}  {ctd_t-t:+10.3f}")
    tuned_rows.append({'dataset': 'SUPPORT', 'model': model,
                       'ctd_default': d, 'ctd_tuned': ctd_t, 'target': t,
                       'gap_default': d - t, 'gap_tuned': ctd_t - t})

# Persist tuned SUPPORT results so the supplementary-table script can pick them
# up alongside output/vig3_jmlr.csv and output/vig3_sksurv.csv.
# 6 decimals lets the script's later 3-dp rounding match a single rounding of
# the raw value (residual error at 6 dp is negligible).
pd.DataFrame(tuned_rows).to_csv("output/vig3_tuned.csv",
                               index=False, float_format="%.6f")

**To reproduce the tuning:**

```python
learner = SurvivalLearner(X_support, y_support)
learner.setup(test_fraction=0.2, cv_folds=5, random_state=42, scoring='ctd')

# Tune each model separately so they get their own Optuna study
for model in ['deepsurv', 'deephit']:
    results = learner.run(model, n_trials=100,
                          experiment_name=f'{model}_support_ctd')
    print(model, results[model].best_params_)
```

The studies are saved to `optuna_studies/<model>_support_ctd/` and can be resumed with `resume=True` for additional trials.

## scikit-survival Benchmarks (Harrell C-index, test set)

scikit-survival's docs report Harrell's C-index on a held-out test set, but each published number is a single split at one seed, so it carries a lot of sampling noise. To compare fairly we average instead of trusting one split: the `target` column below is the **mean of scikit-survival's own recipe over 30 seeds**, and we run mlsurv over the same 30 seeds and report its mean ± std. Mean against mean.

The targets come from `scripts/benchmarks/vignette_03_sksurv_targets.py`. That script first reproduces the exact values printed in the scikit-survival docs (GBSG2 CoxNet 0.688 and RSF 0.692 from the [*Evaluating Survival Models*](https://scikit-survival.readthedocs.io/en/stable/user_guide/evaluating-survival-models.html) guide, on a stratified 75/25 split at seed 1; Veterans FSSVM 0.720 from the [*Survival SVM*](https://scikit-survival.readthedocs.io/en/stable/user_guide/survival-svm.html) guide, the mean of 100 random 50/50 splits), then averages each recipe over the seed list. Re-run it to refresh the targets.

FSSVM's published value is already a 100-split mean, so it is replicated (0.720) rather than re-averaged over our seed list — it has no std.

**Preprocessing**: we match the sksurv docs — one-hot encode categoricals via `sksurv.preprocessing.encode_categorical`, no feature selection.

> **A note on warnings.** GBSG2 and Veterans are small, and several covariates violate the proportional-hazards and log-linearity assumptions, so mlsurv raises a `MethodologicalWarning` (and the occasional `SuggestionWarning`) on essentially every fit. Because the sweep below rebuilds a learner for all 30 seeds of each model, those same notes would otherwise repeat hundreds of times and bury the results. We silence both tiers at the top of the notebook (see the imports cell); a single fit still surfaces them normally. See [`docs/warnings.md`](../docs/warnings.md) to inspect or re-enable them.

In [ ]:
# Targets are scikit-survival's own recipes averaged over the seed list, each
# stored as (mean, std). They come from
# scripts/benchmarks/vignette_03_sksurv_targets.py — re-run it to refresh.
# FSSVM's published value is already a 100-split mean, so it has no std (None).
SKSURV_TARGETS = {
    'GBSG2':    {'coxnet': (0.669, 0.024),
                 'rsf':    (0.691, 0.028)},
    'Veterans': {'fssvm':  (0.720, None)},
}

# Pinned seed list for the mlsurv side. Use >= 30 seeds: the mean's third decimal
# is seed-sensitive at small N (see the targets script). To smoke-test this cell
# quickly before the full run, temporarily set SEEDS = range(2).
SEEDS = range(30)

sksurv_datasets = [
    ('GBSG2',    prepare_gbsg2),
    ('Veterans', prepare_veterans),
]

sksurv_rows = []

for ds_name, loader in sksurv_datasets:
    models = list(SKSURV_TARGETS[ds_name].keys())
    X, y = loader()
    for model in models:
        scores = []
        for seed in SEEDS:
            learner = SurvivalLearner(X, y)
            # setup() prints a banner even with verbose=False, so mute it.
            with contextlib.redirect_stdout(io.StringIO()):
                learner.setup(test_fraction=0.2, cv_folds=5, random_state=seed)
                results = learner.run(models=[model], tune=False, verbose=False,
                                      n_importance_repeats=0, bootstrap=False)
            score = results[model].get_metric('harrell', phase='test')
            # A None here would become NaN and silently poison the mean/gap.
            if score is None:
                raise RuntimeError(
                    f"{ds_name}/{model} seed={seed}: harrell test metric is None; "
                    f"cannot build a benchmark mean from a failed seed.")
            scores.append(score)
        scores = np.array(scores, dtype=float)
        target_mean, target_std = SKSURV_TARGETS[ds_name][model]
        sksurv_rows.append({'dataset': ds_name, 'model': model,
                            'harrell_mean': scores.mean(), 'harrell_std': scores.std(),
                            'target': target_mean, 'target_std': target_std,
                            'gap': scores.mean() - target_mean})

### scikit-survival Summary

In [ ]:
sksurv_df = pd.DataFrame(sksurv_rows)
print("Harrell C-index (test set), mlsurv vs scikit-survival, mean over 30 seeds")
print("-" * 72)
for _, row in sksurv_df.iterrows():
    ts = row['target_std']
    target_str = f"{row['target']:.3f}+/-{ts:.3f}" if pd.notna(ts) else f"{row['target']:.3f} (mean)"
    print(f"  {row['dataset']:10s} {row['model']:8s}  "
          f"{row['harrell_mean']:.3f}+/-{row['harrell_std']:.3f}  "
          f"target={target_str}  gap={row['gap']:+.3f}")
print(f"\nMean |gap|: {sksurv_df['gap'].abs().mean():.3f}")
sksurv_df.to_csv("output/vig3_sksurv.csv", index=False, float_format="%.6f")

## Combined Summary

In [ ]:
combined = pd.concat([
    jmlr_df.rename(columns={'ctd': 'score'})[['dataset', 'model', 'score', 'target', 'gap']].assign(
        suite='JMLR', metric='C^td (CV)'),
    sksurv_df.rename(columns={'harrell_mean': 'score'})[['dataset', 'model', 'score', 'target', 'gap']].assign(
        suite='sksurv', metric='Harrell (test)'),
], ignore_index=True)

print("All benchmarks — default hyperparameters, no tuning")
print("=" * 70)
for suite in ['JMLR', 'sksurv']:
    sub = combined[combined['suite'] == suite]
    metric = sub['metric'].iloc[0]
    print(f"\n  {suite} ({metric})")
    print(f"  {'-' * 62}")
    for _, row in sub.iterrows():
        score_str = f"{row['score']:.3f}" if pd.notna(row['score']) else "N/A"
        sign = "+" if row['gap'] >= 0 else ""
        print(f"    {row['dataset']:10s} {row['model']:12s}  {score_str}  target={row['target']:.3f}  gap={sign}{row['gap']:.3f}")
    print(f"    Mean |gap|: {sub['gap'].abs().mean():.3f}")

print(f"\n  Overall mean |gap|: {combined['gap'].abs().mean():.3f}")

## GPU Acceleration

The deep learning models (DeepSurv, DeepHit) support GPU acceleration via the
`device` parameter. On datasets the size of SUPPORT (n=8,873), GPU provides a
meaningful speedup:

| Model    | CPU fit | GPU fit | Speedup |
|----------|---------|---------|--------:|
| DeepSurv | 11.3 s  | 1.7 s   | ~6.5x   |
| DeepHit  | 13.1 s  | 2.1 s   | ~6.3x   |

*Benchmarked on NVIDIA A30, PyTorch 2.6, CUDA 12.4. Speedup varies with
GPU model, dataset size, and network architecture.*

To enable GPU acceleration, pass `device='cuda'` at setup or training time:

```python
# Option 1: set at setup (applies to all deep learning models)
learner.setup(device='cuda', ...)

# Option 2: set per model at train time
result = learner.train('deepsurv', device='cuda')

# Option 3: auto-detect (default behavior)
# device='auto' uses GPU when available, CPU otherwise
learner.setup(device='auto', ...)
```

**Notes:**
- Non-deep-learning models (CoxPH, RSF, XGBoost, etc.) ignore the `device`
  parameter and always run on CPU.
- Results are deterministic within the same device (same seed = identical output),
  but CPU and GPU may produce slightly different results due to floating-point
  reduction order. This is standard behavior across all deep learning frameworks.
- The speedup grows with dataset size and network complexity. For small datasets
  (n < 1,000) or tiny networks, CPU may be faster due to GPU transfer overhead.

## References

- Curtis C, Shah SP, Chin SF, et al. (2012). The genomic and transcriptomic architecture of 2,000 breast tumours reveals novel subgroups. *Nature*, 486(7403):346–352. https://doi.org/10.1038/nature10983
- Dispenzieri A, Katzmann JA, Kyle RA, et al. (2012). Use of nonclonal serum immunoglobulin free light chains to predict overall survival in the general population. *Mayo Clinic Proceedings*, 87(6):517–523. https://doi.org/10.1016/j.mayocp.2012.03.009
- Kalbfleisch JD, Prentice RL (2002). *The Statistical Analysis of Failure Time Data*, 2nd ed. Wiley. https://doi.org/10.1002/9781118032985
- Katzman JL, Shaham U, Cloninger A, Bates J, Jiang T, Kluger Y (2018). DeepSurv: personalized treatment recommender system using a Cox proportional hazards deep neural network. *BMC Medical Research Methodology*, 18:24. https://doi.org/10.1186/s12874-018-0482-1
- Knaus WA, Harrell FE, Lynn J, et al. (1995). The SUPPORT prognostic model: objective estimates of survival for seriously ill hospitalized adults. *Annals of Internal Medicine*, 122(3):191–203. https://doi.org/10.7326/0003-4819-122-3-199502010-00007
- Kvamme H, Borgan Ø, Scheel I (2019). Time-to-event prediction with neural networks and Cox regression. *Journal of Machine Learning Research*, 20(129):1–30. https://jmlr.org/papers/v20/18-424.html. arXiv:1907.00825
- Kvamme H, Borgan Ø (2021). Continuous and discrete-time survival prediction with neural networks. *Lifetime Data Analysis*, 27(4):710–736. https://doi.org/10.1007/s10985-021-09532-6
- Pölsterl S (2020). scikit-survival: a library for time-to-event analysis built on top of scikit-learn. *Journal of Machine Learning Research*, 21(212):1–6. https://jmlr.org/papers/v21/20-729.html
- Schumacher M, Bastert G, Bojar H, et al. (1994). Randomized 2×2 trial evaluating hormonal treatment and the duration of chemotherapy in node-positive breast cancer patients (German Breast Cancer Study Group). *Journal of Clinical Oncology*, 12(10):2086–2093. https://doi.org/10.1200/JCO.1994.12.10.2086